# Retrieval — Vector Search with FAISS

## Goal
Given a query, find the most relevant chunks from our knowledge base.

## Pipeline
Chunks -> Embed -> Store in FAISS -> Query -> Retrieve Top-K

## What is FAISS?
Facebook AI Similarity Search — a library for efficient vector search.
Instead of comparing query to every chunk one by one,
FAISS indexes vectors for fast nearest-neighbor search.

In [1]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

## 1. Prepare Knowledge Base
Reuse the chunks from ingestion notebook.

In [3]:
import re

# Document
sample_document = """
Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed. 
It focuses on developing computer programs that can access data and use it 
to learn for themselves.

The process begins with observations or data, such as examples, direct 
experience, or instruction. Machine learning algorithms use computational 
methods to learn information directly from data without relying on a 
predetermined equation as a model.

Deep learning is a subfield of machine learning that uses neural networks 
with many layers. These deep neural networks have revolutionized fields such 
as computer vision, natural language processing, and speech recognition.

Natural language processing (NLP) is another important area of machine learning. 
It enables computers to understand, interpret, and generate human language. 
Applications include sentiment analysis, machine translation, and chatbots.

Reinforcement learning is a type of machine learning where an agent learns 
to make decisions by interacting with an environment. The agent receives 
rewards for correct actions and penalties for incorrect ones.
"""

def clean_text(text: str) -> str:
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r" +", " ", text)
    return text.strip()

def chunk_by_sentences(text: str, sentences_per_chunk: int = 3) -> list:
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = " ".join(sentences[i:i + sentences_per_chunk])
        if chunk.strip():
            chunks.append(chunk.strip())
    return chunks

cleaned_doc = clean_text(sample_document)
chunks = chunk_by_sentences(cleaned_doc)

print(f"Total chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}: {chunk[:80]}...")

Total chunks: 4

Chunk 1: Machine learning is a subset of artificial intelligence that enables systems 
to...

Chunk 2: Machine learning algorithms use computational 
methods to learn information dire...

Chunk 3: Natural language processing (NLP) is another important area of machine learning....

Chunk 4: Reinforcement learning is a type of machine learning where an agent learns 
to m...


## 2. Embed Chunks and Build FAISS Index

In [4]:
# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed chunks
print("Embedding chunks...")
chunk_embeddings = model.encode(chunks)
print(f"Embedding shape: {chunk_embeddings.shape}")

# Build FAISS index
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings.astype(np.float32))

print(f"\nFAISS index built.")
print(f"Total vectors in index: {index.ntotal}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding chunks...
Embedding shape: (4, 384)

FAISS index built.
Total vectors in index: 4


## 3. Query the Index
Given a question, find the most relevant chunks.

In [7]:
def retrieve(query: str, index, chunks: list, model, top_k: int = 2) -> list:
    """
    Retrieve top-k most relevant chunks for a query.
    
    Args:
        query: Search query
        index: FAISS index
        chunks: Original text chunks
        model: Embedding model
        top_k: Number of chunks to retrieve
    
    Returns:
        List of (chunk, score) tuples
    """
    query_embedding = model.encode([query]).astype(np.float32)
    distances, indeces = index.search(query_embedding, top_k)

    results = []
    for dist, idx in zip(distances[0], indeces[0]):
        results.append({
            "chunk": chunks[idx],
            "distance": float(dist),
            "chunk_id": int(idx)
        })
    return results

# Test queries
queries = [
    "What is deep learning?",
    "How does reinforcement learning work?",
    "What are NLP applications?",
]

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 50)
    results = retrieve(query, index, chunks, model, top_k=2)
    for i, result in enumerate(results):
        print(f"\nRank {i+1} (distance: {result["distance"]:.4f}):")
        print(result["chunk"][:150])


Query: What is deep learning?
--------------------------------------------------

Rank 1 (distance: 0.4784):
Machine learning algorithms use computational 
methods to learn information directly from data without relying on a 
predetermined equation as a model

Rank 2 (distance: 0.8588):
Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed

Query: How does reinforcement learning work?
--------------------------------------------------

Rank 1 (distance: 0.4514):
Reinforcement learning is a type of machine learning where an agent learns 
to make decisions by interacting with an environment. The agent receives 


Rank 2 (distance: 1.0525):
Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed

Query: What are NLP applications?
--------------------------------------------------

Rank 1 (distance: 

## 4. Key Observations

| Query | Rank 1 Correct? | Note |
|-------|----------------|------|
| Deep learning | No | Deep learning mixed with ML algorithms in same chunk |
| Reinforcement learning | Yes | Dedicated chunk |
| NLP applications | Yes | Dedicated chunk |

## Key Insight
Retrieval quality depends heavily on chunking.
"Deep learning" shares a chunk with "ML algorithms" —
so the retriever cannot distinguish between them.

Better chunking = one topic per chunk = better retrieval.

## Next
Use retrieved chunks as context to generate answers.
-> 03_generation.ipynb